# OmniVoice Story Audio from TXT

Notebook n?y d?ng setting OmniVoice gi?ng notebook worker update v2, nh?ng ch? t?o audio t? nhi?u file `.txt` truy?n, v? d? `bong_hinh_tren_bang_diem.txt`.

Quy tr?nh: upload nhi?u file `.txt` v? m?t file gi?ng m?u, t? nh?n text gi?ng m?u b?ng Whisper, chia truy?n th?nh ?o?n an to?n cho TTS, sinh t?ng ?o?n OmniVoice, gh?p th?nh audio ho?n ch?nh, r?i t?i zip k?t qu?.

Ch? d?ng v?i gi?ng c?a b?n ho?c gi?ng ?? ???c ch? s? h?u cho ph?p. V?o Runtime > Change runtime type > GPU tr??c khi ch?y.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# OmniVoice README recommends torch/torchaudio 2.8.0 CUDA 12.8.
!pip -q install --force-reinstall --no-deps torch==2.8.0+cu128 torchaudio==2.8.0+cu128 torchvision==0.23.0+cu128 --index-url https://download.pytorch.org/whl/cu128
!pip -q install -U git+https://github.com/k2-fsa/OmniVoice.git faster-whisper soundfile


In [ ]:
# Use models already uploaded to Google Drive. This cell does not download model weights.
# Expected Drive folders:
# - MyDrive/models/OmniVoice
# - MyDrive/models/faster-whisper-large-v3
USE_DRIVE_MODELS = True
OMNIVOICE_MODEL_FOLDER = 'OmniVoice'
ASR_MODEL_FOLDER = 'faster-whisper-large-v3'

if USE_DRIVE_MODELS:
    from google.colab import drive
    from pathlib import Path
    import shutil

    drive.mount('/content/drive')
    drive_models_root = Path('/content/drive/MyDrive/models')
    local_models_root = Path('/content/models')

    def copy_drive_model(folder_name: str, required_file: str | None = None) -> str:
        drive_dir = drive_models_root / folder_name
        local_dir = local_models_root / folder_name
        if required_file and not (drive_dir / required_file).exists():
            raise FileNotFoundError(
                'Model not found in Google Drive. Upload the full model folder so this file exists: '
                f'{drive_dir / required_file}'
            )
        if not required_file and (not drive_dir.exists() or not any(drive_dir.iterdir())):
            raise FileNotFoundError(
                'Model folder not found in Google Drive. Upload the full model folder here: '
                f'{drive_dir}'
            )
        if not local_dir.exists():
            print(f'Copying {drive_dir} -> {local_dir}')
            local_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(drive_dir, local_dir)
        else:
            print(f'Using local model copy: {local_dir}')
        return str(local_dir)

    OMNIVOICE_MODEL_PATH = copy_drive_model(OMNIVOICE_MODEL_FOLDER)
    ASR_MODEL_PATH = copy_drive_model(ASR_MODEL_FOLDER, required_file='model.bin')
else:
    OMNIVOICE_MODEL_PATH = 'k2-fsa/OmniVoice'
    ASR_MODEL_PATH = 'medium'

print('OMNIVOICE_MODEL_PATH =', OMNIVOICE_MODEL_PATH)
print('ASR_MODEL_PATH =', ASR_MODEL_PATH)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

TXT_DIR = Path('/content/story_txt_uploads')
OUTPUT_DIR = Path('/content/story_audio')

for folder in [TXT_DIR, OUTPUT_DIR]:
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True, exist_ok=True)

TEXT_EXTENSIONS = {'.txt'}
AUDIO_EXTENSIONS = {'.mp3', '.wav', '.m4a', '.flac', '.ogg'}

print('Upload .txt story file(s) and one reference voice audio file.')
uploaded = files.upload()
REF_AUDIO = None
for name, data in uploaded.items():
    suffix = Path(name).suffix.lower()
    if suffix in TEXT_EXTENSIONS:
        (TXT_DIR / name).write_bytes(data)
    elif suffix in AUDIO_EXTENSIONS:
        ref_path = Path('/content') / name
        ref_path.write_bytes(data)
        REF_AUDIO = str(ref_path)
    else:
        print(f'Skipped unsupported file: {name}')

if not REF_AUDIO:
    raise RuntimeError('Upload one reference audio file, e.g. audio-truyen.mp3')
if not any(TXT_DIR.glob('*.txt')):
    raise RuntimeError('Upload at least one .txt story file.')

print('Reference audio:', REF_AUDIO)
print('TXT files:')
for path in sorted(TXT_DIR.glob('*.txt')):
    print('-', path.name)


In [ ]:
import html
import re
import subprocess
import torch
import unicodedata
from faster_whisper import WhisperModel

REFERENCE_WAV = '/content/ref.wav'
REFERENCE_START = '0'
REFERENCE_DURATION = '8'
LANGUAGE_ID = 'vi'
ASR_MODEL = ASR_MODEL_PATH

# Best-quality default: use 3-10 seconds of clean same-language speech, mono 24 kHz.
# If the source has leading silence, change REFERENCE_START to where speech begins.
subprocess.run([
    'ffmpeg', '-y', '-i', REF_AUDIO,
    '-ss', REFERENCE_START, '-t', REFERENCE_DURATION,
    '-vn', '-af', 'atrim=start=0,asetpts=PTS-STARTPTS,loudnorm=I=-18:TP=-2:LRA=11',
    '-ar', '24000', '-ac', '1',
    REFERENCE_WAV,
], check=True)

print('Reference WAV:', REFERENCE_WAV)

def clean_reference_text(text: str) -> str:
    text = html.unescape(unicodedata.normalize('NFKC', text))
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\{[^{}]*\}', ' ', text)
    text = re.sub(r'["????`]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if text and text[-1] not in '.,!?;:':
        text += '.'
    return text

device = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
asr = WhisperModel(ASR_MODEL, device=device, compute_type=compute_type)
segments, info = asr.transcribe(REFERENCE_WAV, language=LANGUAGE_ID, vad_filter=True)
REF_TEXT = clean_reference_text(' '.join(seg.text.strip() for seg in segments).strip())
print('REF_TEXT =', REF_TEXT)
if not REF_TEXT:
    raise RuntimeError('Whisper did not recognize text from the reference audio. Use a clearer reference clip.')


In [ ]:
# Embedded worker: creates chunked OmniVoice story audio from uploaded TXT files.
from pathlib import Path

WORKER_PATH = '/content/batch_omnivoice_story_audio.py'
Path(WORKER_PATH).write_text('from __future__ import annotations\n\nimport argparse\nimport html\nimport json\nimport logging\nimport re\nimport shutil\nimport subprocess\nimport time\nimport unicodedata\nfrom pathlib import Path\n\nDEFAULT_MAX_CHARS = 520\nDEFAULT_PAUSE_SECONDS = 0.35\nDEFAULT_TRIM_START_SECONDS = 4 / 30\nDEFAULT_TRIM_END_SECONDS = 8 / 30\n\nCASE_SENSITIVE_READINGS = {\n    "AI": "?y ai",\n    "API": "?y pi ai",\n    "CPU": "si pi diu",\n    "GPU": "gi pi diu",\n    "CEO": "si i ?",\n    "USB": "diu ?t bi",\n    "USD": "?? la M?",\n    "USA": "M?",\n    "UK": "Anh",\n    "OK": "? k?",\n}\n\nLETTER_READINGS = {\n    "A": "?y", "B": "bi", "C": "si", "D": "?i", "E": "i", "F": "?p",\n    "G": "gi", "H": "h?t", "I": "ai", "J": "gi?y", "K": "k?y", "L": "eo",\n    "M": "em", "N": "en", "O": "?", "P": "pi", "Q": "kiu", "R": "a",\n    "S": "?t", "T": "ti", "U": "diu", "V": "vi", "W": "??p liu", "X": "?ch",\n    "Y": "goai", "Z": "di",\n}\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Batch OmniVoice story audiobook generation from TXT files.")\n    parser.add_argument("--txt-dir", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--ref-audio", required=True)\n    parser.add_argument("--ref-text", default=None)\n    parser.add_argument("--model", default="k2-fsa/OmniVoice")\n    parser.add_argument("--device", default="cuda:0")\n    parser.add_argument("--dtype", default="float16", choices=["float16", "float32"])\n    parser.add_argument("--speed", type=float, default=1.0)\n    parser.add_argument("--num-step", type=int, default=64)\n    parser.add_argument("--max-chars", type=int, default=DEFAULT_MAX_CHARS)\n    parser.add_argument("--pause-between-chunks", type=float, default=DEFAULT_PAUSE_SECONDS)\n    parser.add_argument("--trim-start-seconds", type=float, default=DEFAULT_TRIM_START_SECONDS)\n    parser.add_argument("--trim-end-seconds", type=float, default=DEFAULT_TRIM_END_SECONDS)\n    parser.add_argument("--output-format", default="mp3", choices=["mp3", "wav", "both"])\n    args = parser.parse_args()\n\n    txt_dir = Path(args.txt_dir)\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    logger = _setup_logger(output_dir / "batch_omnivoice_story_audio.log")\n    _check_binary("ffmpeg")\n    _check_binary("ffprobe")\n\n    txt_files = sorted(txt_dir.glob("*.txt"))\n    if not txt_files:\n        raise RuntimeError(f"No .txt files found in {txt_dir}")\n\n    import soundfile as sf\n    import torch\n    from omnivoice import OmniVoice\n\n    dtype = torch.float16 if args.dtype == "float16" else torch.float32\n    logger.info("Loading OmniVoice model=%s device=%s dtype=%s", args.model, args.device, args.dtype)\n    model = OmniVoice.from_pretrained(args.model, device_map=args.device, dtype=dtype)\n\n    for index, txt_path in enumerate(txt_files, start=1):\n        logger.info("Processing TXT %s/%s: %s", index, len(txt_files), txt_path)\n        _process_one_story(\n            model=model,\n            sf=sf,\n            txt_path=txt_path,\n            ref_audio=Path(args.ref_audio),\n            ref_text=args.ref_text,\n            output_dir=output_dir,\n            speed=args.speed,\n            num_step=args.num_step,\n            max_chars=args.max_chars,\n            pause_between_chunks=args.pause_between_chunks,\n            trim_start_seconds=args.trim_start_seconds,\n            trim_end_seconds=args.trim_end_seconds,\n            output_format=args.output_format,\n            logger=logger,\n        )\n\n    zip_base = output_dir.parent / "omnivoice_story_audio_results"\n    if zip_base.with_suffix(".zip").exists():\n        zip_base.with_suffix(".zip").unlink()\n    shutil.make_archive(str(zip_base), "zip", output_dir)\n    logger.info("Created zip: %s.zip", zip_base)\n\n\ndef _process_one_story(\n    model,\n    sf,\n    txt_path: Path,\n    ref_audio: Path,\n    ref_text: str | None,\n    output_dir: Path,\n    speed: float,\n    num_step: int,\n    max_chars: int,\n    pause_between_chunks: float,\n    trim_start_seconds: float,\n    trim_end_seconds: float,\n    output_format: str,\n    logger: logging.Logger,\n) -> None:\n    name = _safe_stem(txt_path)\n    story_dir = output_dir / name\n    chunk_dir = story_dir / "chunks"\n    chunk_dir.mkdir(parents=True, exist_ok=True)\n\n    raw_text = _read_text(txt_path)\n    paragraphs = _normalize_story_text(raw_text)\n    chunks = _split_story_chunks(paragraphs, max_chars=max_chars)\n    if not chunks:\n        logger.warning("Skipping empty TXT: %s", txt_path)\n        return\n\n    logger.info("Story %s split into %s chunk(s), max_chars=%s", txt_path.name, len(chunks), max_chars)\n    manifest = []\n    audio_paths = []\n\n    for chunk_index, chunk in enumerate(chunks, start=1):\n        tts_text = _prepare_tts_text(chunk)\n        wav_path = chunk_dir / f"{chunk_index:04d}.wav"\n        logger.info(\n            "OmniVoice generate story=%s chunk=%s/%s chars=%s normalized_chars=%s",\n            txt_path.name,\n            chunk_index,\n            len(chunks),\n            len(chunk),\n            len(tts_text),\n        )\n        audio = model.generate(\n            text=tts_text,\n            ref_audio=str(ref_audio),\n            ref_text=ref_text,\n            speed=speed,\n            num_step=num_step,\n        )\n        audio_data = _trim_audio_edges(audio[0], 24000, trim_start_seconds, trim_end_seconds)\n        sf.write(str(wav_path), audio_data, 24000, subtype="PCM_24")\n        audio_paths.append(wav_path)\n        duration = _duration(wav_path, logger)\n        manifest.append({\n            "index": chunk_index,\n            "text": chunk,\n            "tts_text": tts_text,\n            "chars": len(chunk),\n            "tts_chars": len(tts_text),\n            "duration": round(duration, 3),\n            "audio": str(wav_path.relative_to(story_dir)),\n        })\n\n    (story_dir / f"{name}.txt").write_text(raw_text, encoding="utf-8")\n    _write_json(story_dir / "chunks.json", manifest)\n\n    full_wav = story_dir / f"{name}_full.wav"\n    _concat_chunks(audio_paths, full_wav, pause_between_chunks, logger, story_dir)\n\n    if output_format in {"mp3", "both"}:\n        full_mp3 = story_dir / f"{name}_full.mp3"\n        _run([\n            "ffmpeg", "-y", "-i", str(full_wav),\n            "-ar", "44100", "-ac", "2", "-c:a", "libmp3lame", "-b:a", "192k",\n            str(full_mp3),\n        ], logger)\n        logger.info("Finished MP3: %s", full_mp3)\n\n    if output_format == "mp3":\n        full_wav.unlink(missing_ok=True)\n    else:\n        logger.info("Finished WAV: %s", full_wav)\n\n\ndef _read_text(path: Path) -> str:\n    data = path.read_bytes()\n    for encoding in ["utf-8-sig", "utf-16", "cp1258"]:\n        try:\n            return data.decode(encoding)\n        except UnicodeError:\n            continue\n    return data.decode("utf-8", errors="replace")\n\n\ndef _normalize_story_text(text: str) -> list[str]:\n    text = html.unescape(unicodedata.normalize("NFKC", text))\n    text = text.replace("\\r\\n", "\\n").replace("\\r", "\\n")\n    text = re.sub(r"[\\u200b\\ufeff]+", "", text)\n    text = re.sub(r"[ \\t]+", " ", text)\n    paragraphs = [re.sub(r"\\s+", " ", part).strip() for part in re.split(r"\\n\\s*\\n+", text)]\n    return [part for part in paragraphs if part]\n\n\ndef _split_story_chunks(paragraphs: list[str], max_chars: int) -> list[str]:\n    max_chars = max(180, int(max_chars))\n    chunks = []\n    current = ""\n    for paragraph in paragraphs:\n        pieces = _split_long_paragraph(paragraph, max_chars)\n        for piece in pieces:\n            if not current:\n                current = piece\n            elif len(current) + 2 + len(piece) <= max_chars:\n                current = current + "\\n\\n" + piece\n            else:\n                chunks.append(current)\n                current = piece\n    if current:\n        chunks.append(current)\n    return chunks\n\n\ndef _split_long_paragraph(paragraph: str, max_chars: int) -> list[str]:\n    if len(paragraph) <= max_chars:\n        return [paragraph]\n    sentences = [part.strip() for part in re.split(r"(?<=[.!?;:?])\\s+", paragraph) if part.strip()]\n    if len(sentences) <= 1:\n        return _hard_wrap(paragraph, max_chars)\n    result = []\n    current = ""\n    for sentence in sentences:\n        if len(sentence) > max_chars:\n            if current:\n                result.append(current)\n                current = ""\n            result.extend(_hard_wrap(sentence, max_chars))\n        elif not current:\n            current = sentence\n        elif len(current) + 1 + len(sentence) <= max_chars:\n            current += " " + sentence\n        else:\n            result.append(current)\n            current = sentence\n    if current:\n        result.append(current)\n    return result\n\n\ndef _hard_wrap(text: str, max_chars: int) -> list[str]:\n    words = text.split()\n    result = []\n    current = ""\n    for word in words:\n        if not current:\n            current = word\n        elif len(current) + 1 + len(word) <= max_chars:\n            current += " " + word\n        else:\n            result.append(current)\n            current = word\n    if current:\n        result.append(current)\n    return result\n\n\ndef _apply_pronunciation_dictionary(text: str) -> str:\n    for source, target in sorted(CASE_SENSITIVE_READINGS.items(), key=lambda item: len(item[0]), reverse=True):\n        text = re.sub(r"(?<![\\w])" + re.escape(source) + r"(?![\\w])", target, text)\n    return re.sub(\n        r"(?<![\\w])([A-Z]{2,})(?![\\w])",\n        lambda match: " ".join(LETTER_READINGS.get(char, char) for char in match.group(1)),\n        text,\n    )\n\n\ndef _prepare_tts_text(text: str) -> str:\n    text = html.unescape(unicodedata.normalize("NFKC", text))\n    text = re.sub(r"<[^>]+>", " ", text)\n    text = re.sub(r"\\{[^{}]*\\}", " ", text)\n    text = text.replace("?", ", ").replace("...", ", ")\n    replacements = {\n        "%": " ph?n tr?m ",\n        "&": " v? ",\n        "@": " a c?ng ",\n        "#": " s? ",\n        "$": " ?? la ",\n        "+": " c?ng ",\n        "=": " b?ng ",\n        "*": " sao ",\n        "?": " nh?n ",\n        "?": " chia ",\n        "/": " tr?n ",\n        "\\\\": " ",\n        "|": " ",\n        "_": " ",\n        "~": " ",\n        "^": " ",\n    }\n    for source, target in replacements.items():\n        text = text.replace(source, target)\n    text = _apply_pronunciation_dictionary(text)\n    text = re.sub(r"[\\[\\]{}()<>]", ", ", text)\n    text = re.sub(r"[\\"\'????`]+", "", text)\n    text = re.sub(r"\\s*[-??]+\\s*", ", ", text)\n    text = re.sub(r"\\s+", " ", text).strip()\n    if text and text[-1] not in ".,!?;:":\n        text += "."\n    return text\n\n\ndef _trim_audio_edges(audio_data, sample_rate: int, trim_start_seconds: float, trim_end_seconds: float):\n    start_samples = max(0, int(round(sample_rate * max(0.0, trim_start_seconds))))\n    end_samples = max(0, int(round(sample_rate * max(0.0, trim_end_seconds))))\n    if start_samples <= 0 and end_samples <= 0:\n        return audio_data\n    if len(audio_data) <= start_samples + end_samples:\n        return audio_data\n    end_index = len(audio_data) - end_samples if end_samples > 0 else len(audio_data)\n    return audio_data[start_samples:end_index]\n\n\ndef _concat_chunks(audio_paths: list[Path], output_path: Path, pause_seconds: float, logger: logging.Logger, story_dir: Path) -> None:\n    if not audio_paths:\n        raise RuntimeError("No audio chunks to concatenate.")\n    list_path = story_dir / "concat_list.txt"\n    silence_path = story_dir / "_pause.wav"\n    pause_seconds = max(0.0, float(pause_seconds))\n    if pause_seconds > 0:\n        _run([\n            "ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=channel_layout=mono:sample_rate=24000",\n            "-t", f"{pause_seconds:.3f}", "-c:a", "pcm_s24le", str(silence_path),\n        ], logger)\n    lines = []\n    for index, path in enumerate(audio_paths):\n        lines.append(f"file \'{path.resolve().as_posix()}\'")\n        if pause_seconds > 0 and index < len(audio_paths) - 1:\n            lines.append(f"file \'{silence_path.resolve().as_posix()}\'")\n    list_path.write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    _run([\n        "ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(list_path),\n        "-ar", "44100", "-ac", "2", "-c:a", "pcm_s24le", str(output_path),\n    ], logger)\n    list_path.unlink(missing_ok=True)\n    silence_path.unlink(missing_ok=True)\n\n\ndef _duration(path: Path, logger: logging.Logger) -> float:\n    completed = _run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", str(path)], logger)\n    return float(completed.stdout.strip())\n\n\ndef _safe_stem(path: Path) -> str:\n    stem = re.sub(r"[^A-Za-z0-9_.-]+", "_", path.stem).strip("._")\n    return stem or "story"\n\n\ndef _run(cmd: list[str], logger: logging.Logger) -> subprocess.CompletedProcess:\n    logger.info("Running command: %s", " ".join(cmd))\n    completed = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")\n    if completed.stdout.strip():\n        logger.info("stdout: %s", completed.stdout.strip()[-3000:])\n    if completed.stderr.strip():\n        logger.info("stderr: %s", completed.stderr.strip()[-3000:])\n    if completed.returncode != 0:\n        raise RuntimeError(f"Command failed with code {completed.returncode}: {\' \'.join(cmd)}")\n    return completed\n\n\ndef _check_binary(name: str) -> None:\n    if shutil.which(name) is None:\n        raise RuntimeError(f"Missing dependency: {name}. Install ffmpeg and add it to PATH.")\n\n\ndef _setup_logger(log_path: Path) -> logging.Logger:\n    logger = logging.getLogger(f"dichvideo.story_audio.{log_path.parent.name}.{int(time.time())}")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")\n    file_handler = logging.FileHandler(log_path, encoding="utf-8")\n    file_handler.setFormatter(formatter)\n    logger.addHandler(file_handler)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(formatter)\n    logger.addHandler(stream_handler)\n    return logger\n\n\ndef _write_json(path: Path, data) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Worker written to:', WORKER_PATH)


In [ ]:
import torch

MODEL = OMNIVOICE_MODEL_PATH
NUM_STEP = '64'  # Same high-quality default as update_v2.
SPEED = '1.00'  # Vietnamese narration stability.
MAX_CHARS = '520'  # Lower this if OmniVoice misses words; raise slowly if output is stable.
PAUSE_BETWEEN_CHUNKS = '0.35'
TRIM_START_SECONDS = str(4 / 30)  # Same frame trim idea as update_v2: SEGMENT_TRIM_START_FRAMES=4 at 30 fps.
TRIM_END_SECONDS = str(8 / 30)    # Same as update_v2: SEGMENT_TRIM_END_FRAMES=8 at 30 fps.
OUTPUT_FORMAT = 'mp3'  # mp3, wav, or both.
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float16' if torch.cuda.is_available() else 'float32'

!python "$WORKER_PATH"   --txt-dir "$TXT_DIR"   --output-dir "$OUTPUT_DIR"   --ref-audio "$REFERENCE_WAV"   --ref-text "$REF_TEXT"   --model "$MODEL"   --device "$DEVICE"   --dtype "$DTYPE"   --speed "$SPEED"   --num-step "$NUM_STEP"   --max-chars "$MAX_CHARS"   --pause-between-chunks "$PAUSE_BETWEEN_CHUNKS"   --trim-start-seconds "$TRIM_START_SECONDS"   --trim-end-seconds "$TRIM_END_SECONDS"   --output-format "$OUTPUT_FORMAT"


In [ ]:
from google.colab import files
from pathlib import Path

zip_path = Path('/content/omnivoice_story_audio_results.zip')
if zip_path.exists():
    files.download(str(zip_path))
else:
    print('No story audio zip found.')
